In [ ]:
# %% [code] {"execution":{"iopub.status.busy":"2026-09-13T19:08:35.986369Z","iopub.execute_input":"2026-09-13T19:08:35.986876Z","iopub.status.idle":"2026-09-13T19:08:35.993776Z","shell.execute_reply.started":"2026-09-13T19:08:35.986842Z","shell.execute_reply":"2026-09-13T19:08:35.992817Z"},"jupyter":{"outputs_hidden":false}}
import sys
import numpy as np
import pandas as pd
import sklearn
import scipy
import torch

print("Python       :", sys.version)
print("NumPy        :", np.__version__)
print("Pandas       :", pd.__version__)
print("Scikit-learn :", sklearn.__version__)
print("SciPy        :", scipy.__version__)
print("PyTorch      :", torch.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# %% [code] {"execution":{"iopub.status.busy":"2026-09-13T19:08:35.994892Z","iopub.execute_input":"2026-09-13T19:08:35.995245Z","iopub.status.idle":"2026-09-13T19:08:39.730967Z","shell.execute_reply.started":"2026-09-13T19:08:35.995212Z","shell.execute_reply":"2026-09-13T19:08:39.729784Z"},"jupyter":{"outputs_hidden":false}}
!pip install -q sentence-transformers datasets

# %% [code] {"execution":{"iopub.status.busy":"2026-09-13T19:08:39.732999Z","iopub.execute_input":"2026-09-13T19:08:39.733387Z","iopub.status.idle":"2026-09-13T19:08:39.740064Z","shell.execute_reply.started":"2026-09-13T19:08:39.733356Z","shell.execute_reply":"2026-09-13T19:08:39.739234Z"},"jupyter":{"outputs_hidden":false}}
import torch
import transformers
import sentence_transformers
import datasets

from sentence_transformers import SentenceTransformer
from datasets import load_dataset

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("Datasets:", datasets.__version__)

print("\nAll RAG libraries imported successfully.")

# %% [code] {"execution":{"iopub.status.busy":"2026-09-13T19:08:39.741132Z","iopub.execute_input":"2026-09-13T19:08:39.741434Z","iopub.status.idle":"2026-09-13T19:08:40.263545Z","shell.execute_reply.started":"2026-09-13T19:08:39.741401Z","shell.execute_reply":"2026-09-13T19:08:40.262665Z"},"jupyter":{"outputs_hidden":false}}
from datasets import load_dataset

dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
)

dataset

In [86]:
from datasets import load_dataset
import pandas as pd
import numpy as np

# Load the original dataset
dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
)

# Convert train split to DataFrame
df = dataset["train"].to_pandas()

print("Dataset loaded successfully.")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nFirst 5 examples:")
display(df.head())


Dataset loaded successfully.
Shape: (26872, 5)

Columns:
['flags', 'instruction', 'category', 'intent', 'response']

Missing values:
flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64

First 5 examples:


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [87]:
# Remove duplicate customer instructions

original_rows = len(df)
unique_instructions = df["instruction"].nunique()

rag_df = (
    df[
        ["instruction", "category", "intent", "response"]
    ]
    .drop_duplicates(
        subset=["instruction"],
        keep="first"
    )
    .reset_index(drop=True)
)

print("Original rows      :", original_rows)
print("Unique instructions:", unique_instructions)
print("Removed duplicates :", original_rows - len(rag_df))
print("Final RAG dataset  :", len(rag_df))

print("\nUnique intent count:", rag_df["intent"].nunique())


Original rows      : 26872
Unique instructions: 24635
Removed duplicates : 2237
Final RAG dataset  : 24635

Unique intent count: 27


In [88]:
from sklearn.model_selection import train_test_split

# Split the deduplicated RAG dataset
rag_train_df, rag_eval_df = train_test_split(
    rag_df,
    test_size=0.20,
    random_state=42,
    stratify=rag_df["intent"]
)

# Reset indices
rag_train_df = rag_train_df.reset_index(drop=True)
rag_eval_df = rag_eval_df.reset_index(drop=True)

print("RAG train samples:", len(rag_train_df))
print("RAG evaluation samples:", len(rag_eval_df))

print("\nTrain intent distribution:")
print(rag_train_df["intent"].value_counts().sort_index())

print("\nEvaluation intent distribution:")
print(rag_eval_df["intent"].value_counts().sort_index())


RAG train samples: 19708
RAG evaluation samples: 4927

Train intent distribution:
intent
cancel_order                394
change_order                696
change_shipping_address     778
check_cancellation_fee      760
check_invoice               737
check_payment_methods       799
check_refund_policy         798
complaint                   800
contact_customer_service    800
contact_human_agent         799
create_account              714
delete_account              734
delivery_options            528
delivery_period             799
edit_account                622
get_invoice                 727
get_refund                  734
newsletter_subscription     799
payment_issue               799
place_order                 798
recover_password            796
registration_problems       799
review                      798
set_up_shipping_address     798
switch_account              690
track_order                 646
track_refund                566
Name: count, dtype: int64

Evaluation intent di

In [89]:
# Final data leakage check

train_instructions = set(rag_train_df["instruction"])
eval_instructions = set(rag_eval_df["instruction"])

overlap = train_instructions.intersection(eval_instructions)

print("Data Leakage Check")
print("=" * 60)
print("RAG train samples :", len(rag_train_df))
print("RAG eval samples  :", len(rag_eval_df))

print("\nUnique train instructions:", len(train_instructions))
print("Unique eval instructions :", len(eval_instructions))

print("\nExact overlapping instructions:", len(overlap))

leakage_rate = len(overlap) / len(eval_instructions) * 100

print("Leakage rate:", f"{leakage_rate:.4f}%")

if len(overlap) == 0:
    print("\nPASS: No exact instruction leakage detected.")
else:
    print("\nWARNING: Exact instruction overlap detected.")
    print("\nExamples of overlapping instructions:")
    
    for instruction in list(overlap)[:10]:
        print("-", instruction)


Data Leakage Check
RAG train samples : 19708
RAG eval samples  : 4927

Unique train instructions: 19708
Unique eval instructions : 4927

Exact overlapping instructions: 0
Leakage rate: 0.0000%

PASS: No exact instruction leakage detected.


In [90]:
import torch
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Embedding model loaded successfully.")
print("Model:", EMBEDDING_MODEL_NAME)
print("Device:", embedding_model.device)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded successfully.
Model: all-MiniLM-L6-v2
Device: cuda:0


In [91]:
# Generate embeddings for RAG training documents only

rag_documents = rag_train_df["instruction"].astype(str).tolist()

print("Number of RAG training documents:", len(rag_documents))

rag_train_embeddings = embedding_model.encode(
    rag_documents,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

print("\nEmbeddings generated successfully.")
print("Shape:", rag_train_embeddings.shape)
print("Dtype:", rag_train_embeddings.dtype)


Number of RAG training documents: 19708


Batches:   0%|          | 0/308 [00:00<?, ?it/s]


Embeddings generated successfully.
Shape: (19708, 384)
Dtype: float32


In [92]:
import faiss

# Create FAISS index
embedding_dimension = rag_train_embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dimension)

# Add RAG training embeddings only
index.add(rag_train_embeddings)

print("FAISS index created successfully.")
print("Embedding dimension:", embedding_dimension)
print("Number of vectors:", index.ntotal)


FAISS index created successfully.
Embedding dimension: 384
Number of vectors: 19708


In [93]:
# Verify alignment between RAG training data and embeddings

print("FAISS vectors :", index.ntotal)
print("RAG train rows:", len(rag_train_df))
print("Embeddings rows:", len(rag_train_embeddings))

assert index.ntotal == len(rag_train_df)
assert len(rag_train_embeddings) == len(rag_train_df)

print("\nAlignment check passed.")
print("Each FAISS vector corresponds to the same row in rag_train_df.")


FAISS vectors : 19708
RAG train rows: 19708
Embeddings rows: 19708

Alignment check passed.
Each FAISS vector corresponds to the same row in rag_train_df.


In [94]:
# RAG retrieval function

def retrieve_response(query, top_k=3):
    """
    Retrieve the most relevant responses from the RAG training set.
    """

    # Encode query
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype("float32")

    # Search FAISS
    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "instruction": rag_train_df.iloc[idx]["instruction"],
            "category": rag_train_df.iloc[idx]["category"],
            "intent": rag_train_df.iloc[idx]["intent"],
            "response": rag_train_df.iloc[idx]["response"]
        })

    return results

print("Retrieval function created successfully.")


Retrieval function created successfully.


In [95]:
# Test retrieval with a single query

query = "I want to cancel my order"

results = retrieve_response(query, top_k=3)

print("=" * 100)
print("QUERY:")
print(query)

for rank, result in enumerate(results, start=1):
    print("\n" + "-" * 100)
    print("Rank    :", rank)
    print("Score   :", round(result["score"], 4))
    print("Intent  :", result["intent"])
    print("Category:", result["category"])
    print("Text    :", result["instruction"])
    print("Response:", result["response"][:300], "...")


QUERY:
I want to cancel my order

----------------------------------------------------------------------------------------------------
Rank    : 1
Score   : 0.891
Intent  : cancel_order
Category: ORDER
Text    : canceling order
Response: I pick up what you're putting down, your need to cancel your order. Let's make this process as smooth as possible. Here's what you need to do:

1. Log in to your {{Online Company Portal Info}}.
2. Navigate to the '{{Online Order Interaction}}' or '{{Online Order Interaction}}' section.
3. Locate the ...

----------------------------------------------------------------------------------------------------
Rank    : 2
Score   : 0.8757
Intent  : cancel_order
Category: ORDER
Text    : I bought some product, I want to cancel order {{Order Number}}
Response: I've come to understand that you've made a purchase and now you wish to cancel the order with the order number {{Order Number}}. Our cancellations team is here to help you with that. To proceed with the c

In [96]:
# Evaluate RAG retrieval on the leakage-free evaluation set

def evaluate_retrieval(eval_df, top_k=5):
    hits_at_1 = 0
    hits_at_3 = 0
    hits_at_5 = 0

    total = len(eval_df)

    for _, row in eval_df.iterrows():
        query = row["instruction"]
        true_intent = row["intent"]

        # Encode evaluation query
        query_embedding = embedding_model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        ).astype("float32")

        # Retrieve from RAG TRAIN index only
        scores, indices = index.search(query_embedding, top_k)

        retrieved_intents = [
            rag_train_df.iloc[idx]["intent"]
            for idx in indices[0]
        ]

        if true_intent == retrieved_intents[0]:
            hits_at_1 += 1

        if true_intent in retrieved_intents[:3]:
            hits_at_3 += 1

        if true_intent in retrieved_intents[:5]:
            hits_at_5 += 1

    return {
        "Recall@1": hits_at_1 / total,
        "Recall@3": hits_at_3 / total,
        "Recall@5": hits_at_5 / total
    }


retrieval_results = evaluate_retrieval(
    rag_eval_df,
    top_k=5
)

print("RAG Retrieval Evaluation")
print("=" * 60)
print("Evaluation samples:", len(rag_eval_df))

for metric, value in retrieval_results.items():
    print(f"{metric}: {value:.4f}")


RAG Retrieval Evaluation
Evaluation samples: 4927
Recall@1: 0.9901
Recall@3: 0.9976
Recall@5: 0.9984


In [97]:
# Analyze cases where the top-1 retrieved intent is incorrect

errors = []

for _, row in rag_eval_df.iterrows():
    query = row["instruction"]
    true_intent = row["intent"]

    # Encode query
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype("float32")

    # Retrieve top 5
    scores, indices = index.search(query_embedding, 5)

    retrieved_intents = [
        rag_train_df.iloc[idx]["intent"]
        for idx in indices[0]
    ]

    top1_intent = retrieved_intents[0]

    if top1_intent != true_intent:
        errors.append({
            "query": query,
            "true_intent": true_intent,
            "top1_intent": top1_intent,
            "top1_score": float(scores[0][0]),
            "top3_intents": retrieved_intents[:3]
        })

print("Recall@1 Error Analysis")
print("=" * 70)
print("Total evaluation samples:", len(rag_eval_df))
print("Top-1 errors:", len(errors))
print(
    "Observed Recall@1:",
    round(1 - len(errors) / len(rag_eval_df), 4)
)

print("\nFirst 20 errors:")
for i, error in enumerate(errors[:20], start=1):
    print("\n" + "-" * 70)
    print("Error", i)
    print("Query       :", error["query"])
    print("True intent :", error["true_intent"])
    print("Top-1 intent:", error["top1_intent"])
    print("Top-1 score :", round(error["top1_score"], 4))
    print("Top-3       :", error["top3_intents"])


Recall@1 Error Analysis
Total evaluation samples: 4927
Top-1 errors: 49
Observed Recall@1: 0.9901

First 20 errors:

----------------------------------------------------------------------
Error 1
Query       : i try to sap some items of order {{Order Number}}
True intent : change_order
Top-1 intent: track_order
Top-1 score : 0.7101
Top-3       : ['track_order', 'change_order', 'change_order']

----------------------------------------------------------------------
Error 2
Query       : help creating account
True intent : registration_problems
Top-1 intent: create_account
Top-1 score : 0.9145
Top-3       : ['create_account', 'registration_problems', 'registration_problems']

----------------------------------------------------------------------
Error 3
Query       : I lost bill #85632, could you find it for me?
True intent : check_invoice
Top-1 intent: get_invoice
Top-1 score : 0.9226
Top-3       : ['get_invoice', 'get_invoice', 'check_invoice']

-----------------------------------------